# Libraries management

In [ ]:
import os
import sys
import pandas as pd
## Check python version
import platform

sys.path.insert(0, os.getcwd())      # lets nbconvert / VS Code import auto_lib
import auto_lib as al

In [ ]:
# print(sys.version)
# print(platform.python_version())
# print(al.__version__, pd.__version__)

# Config management

In [ ]:
instant_client_path = r"D:\oracle\instantclient_19_29"

al.init_oracle_client_once(instant_client_path)   # Thick mode; safe to re-run this cell
env = al.load_env()                               # DWH_* from repo root .env, never printed

# Declaration of variables

In [ ]:
sql_script = """
-- Tet windows are written once, as ANSI DATE literals (independent of the session NLS_DATE_FORMAT).
WITH base AS (
    SELECT
        sale_date,
        supplier_code,
        supplier_name,
        dimension_group,
        dimension,
        net_sales,
        ord_cnt,
        byr_cnt,
        margin,
        CASE
            WHEN sale_date BETWEEN DATE '2024-12-15' AND DATE '2025-02-13' THEN 'Tet 2025'
            WHEN sale_date BETWEEN DATE '2026-01-03' AND DATE '2026-03-04' THEN 'Tet 2026'
            ELSE 'Other Period'
        END AS campaign_period
    FROM
        crv_data.loutruong_supplier_perf_di
    WHERE
        1 = 1
        AND (
            sale_date BETWEEN DATE '2024-12-15' AND DATE '2025-02-13'
            OR sale_date BETWEEN DATE '2026-01-03' AND DATE '2026-03-04'
        )
        AND supplier_code IN (
            SELECT
                supplier_code
            FROM
                omni_digimgr.loutruong_dim_supplier
        )
)
SELECT
    base.*,
    DENSE_RANK() OVER (
        PARTITION BY campaign_period
        ORDER BY TRUNC(sale_date)
    ) AS day_number
FROM
    base
ORDER BY
    sale_date ASC,
    supplier_code ASC,
    dimension_group ASC,
    dimension ASC
"""
sheet_path = r"D:\OneDrive - Central Group\Stella's files - 1. HAND OVER\03. REPORT DAILY\09_supplier_tracker\Supplier_Performance_Tracker.xlsx"
sheet_name = 'perf_raw_di_tet'
header_aliases = {'NET_SALES': 'NET_SALE', 'MARGIN': 'FRONT_MARGIN'}   # query column -> sheet header text
allow_empty_result = False     # False = stop before Excel is touched on 0 rows
fetch_arraysize = 10000        # rows per round trip from Oracle

# Tuning knobs live in auto_lib/excel.py ExcelJob, passed as keyword args, e.g. al.ExcelJob(..., refresh_timeout=900)
# Defaults: open_retries=5, open_retry_wait=15, load_wait=3, paste_chunk_rows=50000, strict_header_check=False, refresh_after_paste=True, refresh_timeout=600, refresh_settle_wait=2, excel_exit_timeout=120
job = al.ExcelJob(sheet_path=sheet_path, sheet_name=sheet_name, header_aliases=header_aliases)

# 3. Connection controller

In [ ]:
df = al.fetch_dataframe(sql_script, arraysize=fetch_arraysize, allow_empty=allow_empty_result, env=env)

In [ ]:
# print(df.head(10).to_string())

# print(df.dtypes)

# Main logic: Open > Delete > Write > Refresh > Save > Close

In [ ]:
rows_processed = al.paste_to_sheet(df, job)